# Notebook 05 — Locked test evaluation, advanced analyses and 20 figures

**Purpose.** One-time evaluator-role scoring of the frozen inference bundles on the locked test partition: raw and temperature-calibrated probabilities, patient/row KL, hard metrics, calibration, disagreement and referral, paired component bootstrap, strata, the six-condition stress suite, evidence-deletion audits, latency, resources, and figures V01–V20. **Inputs:** frozen final runs, protocol lock. **Outputs:** `private/evaluation/<protocol_hash>/`, `results/aggregate/`, `figures/`.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
REPO = Path.cwd().resolve() if (Path.cwd() / "src" / "cape_eeg").exists() else Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("CAPE_ROOT", str(REPO.parent)); os.environ.setdefault("HMS_DATA_ROOT", os.environ["CAPE_ROOT"])
os.environ["PYTHONWARNINGS"] = "ignore"
from cape_eeg.paths import resolve_workspace, redact
from cape_eeg.status import read_json, Ledger
ws = resolve_workspace()
def run(cmd, **kw):
    """Run a repository script as a bounded subprocess; prints filtered output (no secrets, no identifiers)."""
    p = subprocess.run([sys.executable, str(REPO / "scripts" / cmd[0]), *cmd[1:]], capture_output=True, text=True, env=os.environ, **kw)
    for line in (p.stdout + p.stderr).splitlines():
        if line.strip() and not any(w in line for w in ("Warning", "warn", "Found GPU", "Minimum and", "(8.0)")):
            print(line)
    if p.returncode != 0:
        raise RuntimeError(f"{cmd[0]} exited with {p.returncode}")
print("repo:", redact(REPO, ws)); print("workspace root:", redact(ws.root, ws)); print("data root:", redact(ws.data, ws)); print("private:", redact(ws.private, ws))


## Final-test gate and one-time evaluation

In [ ]:
lock = read_json(ws.manifests / 'protocol_lock.json'); out = ws.evaluation / lock['protocol_hash']
if not (out / 'summary.json').exists():
    run(['final_evaluate.py'])
else:
    print('locked evaluation already exists for protocol', lock['protocol_hash'], '- not re-run')
print(open(ws.manifests / 'test_access_log.jsonl').read()[:600])

## Primary result

In [ ]:
s = read_json(out / 'summary.json'); pr = s['primary']['primary_calibrated']
print('Delta patient/component-mean KL (P - %s), three seeds, calibrated: %+.4f [%+.4f, %+.4f] -> %s' % (lock['comparator'], pr['point_estimate'], pr['ci_low'], pr['ci_high'], pr['decision']))
print('per-seed P:', [round(x, 4) for x in pr['per_seed_a']], '| comparator:', [round(x, 4) for x in pr['per_seed_b']], '| relative change %.1f%%' % (100 * pr['relative_change']))
import pandas as pd; print(pd.read_csv(ws.results_public / 'table2_test_metrics_summary.csv').to_string(index=False))

## Strata, robustness, evidence and referral

In [ ]:
print(pd.read_csv(ws.results_public / 'table4_strata_robustness_referral.csv').to_string(index=False, max_colwidth=60))

## Twenty figures

In [ ]:
run(['make_figures.py'])
man = read_json(ws.figures_public / 'manifest.json'); print({f['figure_id']: f['status'] for f in man})

## Claims checklist

In [ ]:
print('Primary decision:', pr['decision']); print('Confirmatory comparison count: 1 (P vs locked comparator). All other comparisons are exploratory or development-only.')
print('Resources:', {k: s['resources'][k] for k in ['gpu_hours_total','n_gpu_jobs','failed_or_incomplete_jobs']})